# 05 · Common Crawl, Step 2: from number to company (WET files)

Research log: [Step 6](../docs/research_log.md#step-6--common-crawl-pasul-2-pornind-de-la-număr) (in Romanian). Data and checks as of October 2026.

**Question.** If any UK VAT number is extracted from arbitrary web pages, how often is it correct, and can its owner be found?

**Method.** 3 random WET files out of 100,000 in `CC-MAIN-2026-39` (seed 42); regex + mod-97 checksum; CH number from the surrounding text; first 31 numbers in random order checked by hand at HMRC; attribution by the normalised HMRC name looked up in Companies House.

**Result (from the research log).**
- 64,082 pages → 165 matches → 121 pass the checksum → 108 unique numbers.
- 31 checked at HMRC: 30 valid, 1 invalid (a number published by the company itself).
- 30/30 valid numbers attributed to a single CH company via the HMRC name; the CH number on the page pointed to a different group company in 2/21.

**Notes for re-running.**
- Downloads the 3 WET files (~65 MB each) into `../data/wet/` if missing, and reads the raw CH file.
- HMRC results were transcribed into `data/hmrc_checks/` by hand (see `data/README.md`).

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [7]:
import gzip, io, random, re, time, os
from pathlib import Path
from urllib.parse import urlparse

import requests
import pandas as pd
from warcio.archiveiterator import ArchiveIterator

CRAWL = "CC-MAIN-2026-39"
N_FISIERE = 3
SEED = 42
BAZA = "https://data.commoncrawl.org/"
DIR_WET = Path("../data/wet")
DIR_WET.mkdir(parents=True, exist_ok=True)
HEADERS = {"User-Agent": "veridion-vat-poc (student research)"}

In [8]:
r = requests.get(f"{BAZA}crawl-data/{CRAWL}/wet.paths.gz", headers=HEADERS, timeout=60)
r.raise_for_status()
toate_caile = gzip.decompress(r.content).decode().split()
print("Fișiere WET în crawl:", len(toate_caile))

rng = random.Random(SEED)
alese = rng.sample(toate_caile, N_FISIERE)
pd.DataFrame({"cale": alese}).to_csv("../data/samples/cc_pas2_fisiere_alese.csv", index=False)
for c in alese:
    print(c)

Fișiere WET în crawl: 100000
crawl-data/CC-MAIN-2026-39/segments/1788492992222.68/wet/CC-MAIN-20260915021126-20260915051126-00810.warc.wet.gz
crawl-data/CC-MAIN-2026-39/segments/1788492699655.55/wet/CC-MAIN-20260906075737-20260906105737-00592.warc.wet.gz
crawl-data/CC-MAIN-2026-39/segments/1788492699512.23/wet/CC-MAIN-20260904222409-20260905012409-00278.warc.wet.gz


In [9]:
def descarca(cale, incercari=6):
    dest = DIR_WET / Path(cale).name
    if dest.exists() and dest.stat().st_size > 0:
        return dest, 0.0
    t0 = time.time()
    for i in range(incercari):
        try:
            with requests.get(BAZA + cale, headers=HEADERS, stream=True, timeout=120) as resp:
                if resp.status_code in (503, 502, 504, 429):
                    raise requests.HTTPError(f"status {resp.status_code}")
                resp.raise_for_status()
                tmp = dest.with_suffix(".part")
                with open(tmp, "wb") as f:
                    for bucata in resp.iter_content(chunk_size=1 << 20):
                        f.write(bucata)
                tmp.rename(dest)
                return dest, time.time() - t0
        except (requests.RequestException, OSError) as e:
            asteptare = 2 ** i * 5
            print(f"  eroare ({e}), reîncerc în {asteptare}s")
            time.sleep(asteptare)
    raise RuntimeError(f"Nu am putut descărca {cale}")

timpi_descarcare = {}
for cale in alese:
    dest, sec = descarca(cale)
    timpi_descarcare[cale] = sec
    print(f"{dest.name}: {dest.stat().st_size/1e6:.1f} MB, {sec:.0f}s")

CC-MAIN-20260915021126-20260915051126-00810.warc.wet.gz: 67.3 MB, 31s
CC-MAIN-20260906075737-20260906105737-00592.warc.wet.gz: 63.8 MB, 17s
CC-MAIN-20260904222409-20260905012409-00278.warc.wet.gz: 63.9 MB, 19s


In [12]:
# Checksum TVA UK [din memorie]: ponderi 8..2 pe primele 7 cifre, plus ultimele 2 cifre.
# Valid dacă suma % 97 == 0 sau (suma + 55) % 97 == 0.
def checksum_ok(nr9):
    if len(nr9) != 9 or not nr9.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr9[:7], range(8, 1, -1))) + int(nr9[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

NOUA = r"((?:\d[\s\-]?){8}\d)(?![\s\-]?\d)"       # 9 cifre, cu spații sau cratime între ele
RE_VAT = re.compile(r"\bVAT\b[^0-9]{0,40}?(?:GB)?[\s\-]?" + NOUA, re.I)
RE_GB = re.compile(r"\bGB[\s\-]?" + NOUA)
RE_VAT_CUVANT = re.compile(r"\bvat\b")            # cuvântul întreg, nu subșir („private”)
RE_CH = re.compile(
    r"(?:company|registration|registered|reg\.?)\s*(?:no\.?|number|nr\.?)?[^0-9A-Za-z]{0,20}"
    r"((?:SC|NI|OC|SO|NC|SL|R0)?\d{6,8})\b", re.I)

def extrage(text):
    rez, spans_vat = [], []
    for m in RE_VAT.finditer(text):
        spans_vat.append(m.span(1))
        rez.append(("vat_kw", m))
    for m in RE_GB.finditer(text):
        if any(a <= m.start(1) < b for a, b in spans_vat):
            continue                                   # deja prins de tiparul cu „VAT”
        rez.append(("gb_fara_vat", m))
    return rez

# test regex
for t in ["VAT No: GB 811 0372 80", "VAT Reg. No. 211105734", "Tel 01234 567890 VAT",
          "prices excl. VAT 123456789012"]:
    print(t, "->", [(tip, re.sub(r'\D', '', m.group(1))) for tip, m in extrage(t)])

# test cuvânt „VAT”
for t in ["private elevator", "Prices exclude VAT.", "vat-registered"]:
    print(repr(t), "->", bool(RE_VAT_CUVANT.search(t.lower())))

# test checksum pe cele 13 numere confirmate (completează lista, ca text)
confirmate_13 = []
print("13 confirmate:", all(checksum_ok(n) for n in confirmate_13), len(confirmate_13),
      "| 123456789:", checksum_ok("123456789"))

VAT No: GB 811 0372 80 -> [('vat_kw', '811037280')]
VAT Reg. No. 211105734 -> [('vat_kw', '211105734')]
Tel 01234 567890 VAT -> []
prices excl. VAT 123456789012 -> []
'private elevator' -> False
'Prices exclude VAT.' -> True
'vat-registered' -> True
13 confirmate: True 0 | 123456789: False


In [13]:
candidati, statistici = [], []

for cale in alese:
    fisier = DIR_WET / Path(cale).name
    t0 = time.time()
    st = {"fisier": fisier.name, "mb": round(fisier.stat().st_size / 1e6, 1),
          "sec_descarcare": round(timpi_descarcare.get(cale, 0)),
          "pagini": 0, "pagini_eng": 0, "pagini_uk": 0, "pagini_cu_cuvant_vat": 0,
          "potriviri": 0, "potriviri_checksum_ok": 0}
    with open(fisier, "rb") as f:
        for rec in ArchiveIterator(f):
            if rec.rec_type != "conversion":
                continue
            st["pagini"] += 1
            url = rec.rec_headers.get_header("WARC-Target-URI") or ""
            limba = rec.rec_headers.get_header("WARC-Identified-Content-Language") or ""
            host = urlparse(url).netloc.lower()
            if "eng" in limba:
                st["pagini_eng"] += 1
            if host.endswith(".uk"):
                st["pagini_uk"] += 1
            text = rec.content_stream().read().decode("utf-8", errors="replace")
            low = text.lower()
            if RE_VAT_CUVANT.search(low):
                st["pagini_cu_cuvant_vat"] += 1
            elif "gb" not in low:
                continue
            for tip, m in extrage(text):
                nr = re.sub(r"\D", "", m.group(1))
                ok = checksum_ok(nr)
                st["potriviri"] += 1
                st["potriviri_checksum_ok"] += ok
                ctx = text[max(0, m.start() - 200): m.end() + 200].replace("\n", " ")
                ch = RE_CH.search(ctx)
                candidati.append({
                    "fisier": fisier.name, "url": url, "host": host,
                    "uk": host.endswith(".uk"), "limba": limba, "tip": tip,
                    "vat": nr, "checksum_ok": ok,
                    "ch_in_context": ch.group(1).upper() if ch else "",
                    "context": ctx})
    st["sec_procesare"] = round(time.time() - t0)
    statistici.append(st)
    print(st)

df = pd.DataFrame(candidati)
df_st = pd.DataFrame(statistici)
df.to_csv("../data/results/cc_pas2_candidati.csv", index=False)
df_st.to_csv("../data/results/cc_pas2_statistici.csv", index=False)

{'fisier': 'CC-MAIN-20260915021126-20260915051126-00810.warc.wet.gz', 'mb': 67.3, 'sec_descarcare': 31, 'pagini': 21335, 'pagini_eng': 13554, 'pagini_uk': 423, 'pagini_cu_cuvant_vat': 230, 'potriviri': 58, 'potriviri_checksum_ok': 41, 'sec_procesare': 17}
{'fisier': 'CC-MAIN-20260906075737-20260906105737-00592.warc.wet.gz', 'mb': 63.8, 'sec_descarcare': 17, 'pagini': 21233, 'pagini_eng': 13444, 'pagini_uk': 428, 'pagini_cu_cuvant_vat': 230, 'potriviri': 56, 'potriviri_checksum_ok': 44, 'sec_procesare': 16}
{'fisier': 'CC-MAIN-20260904222409-20260905012409-00278.warc.wet.gz', 'mb': 63.9, 'sec_descarcare': 19, 'pagini': 21514, 'pagini_eng': 13624, 'pagini_uk': 451, 'pagini_cu_cuvant_vat': 216, 'potriviri': 51, 'potriviri_checksum_ok': 36, 'sec_procesare': 16}


In [14]:
print(df_st.sum(numeric_only=True))
print()
print(df.groupby(["tip", "checksum_ok"]).size())
print()
valide = df[df.checksum_ok]
print("Numere unice cu checksum OK:", valide.vat.nunique())
print("…din care pe host .uk:", valide[valide.uk].vat.nunique())
print("…cu număr CH în context:", valide[valide.ch_in_context != ""].vat.nunique())
print("Host-uri distincte cu numere valide:", valide.host.nunique())
valide.drop_duplicates("vat").head(20)[["host", "tip", "vat", "ch_in_context", "url"]]

mb                         195.0
sec_descarcare              67.0
pagini                   64082.0
pagini_eng               40622.0
pagini_uk                 1302.0
pagini_cu_cuvant_vat       676.0
potriviri                  165.0
potriviri_checksum_ok      121.0
sec_procesare               49.0
dtype: float64

tip          checksum_ok
gb_fara_vat  False            5
             True             5
vat_kw       False           39
             True           116
dtype: int64

Numere unice cu checksum OK: 108
…din care pe host .uk: 61
…cu număr CH în context: 74
Host-uri distincte cu numere valide: 106


,host,tip,vat,ch_in_context,url
0,www.buysheds.com,vat_kw,197730860,,http://www.buysheds.com/cat/8-x-4-workshops/
3,adassoc.org.uk,gb_fara_vat,238540264,,https://adassoc.org.uk/resources/creative-access/
5,biology.ed.ac.uk,vat_kw,592950700,SC005336,https://biology.ed.ac.uk/ccbio/our-research/ce...
7,digital.bsa.ac.uk,vat_kw,232298369,,https://digital.bsa.ac.uk/results.php?map-cont...
8,hornbeckgroup.com,vat_kw,113152467,07617882,https://hornbeckgroup.com/product/plymouth-hea...
10,jackspratt.co.uk,vat_kw,168384471,09474102,https://jackspratt.co.uk/christmas-pyjamas-23-...
13,nurserymanagementshow.co.uk,vat_kw,629547604,,https://nurserymanagementshow.co.uk/speakers/t...
15,packaging2yourdoor.co.uk,vat_kw,363459673,,https://packaging2yourdoor.co.uk/shop/1-x-roll...
16,retailrevealed.com,vat_kw,843845601,00454825,https://retailrevealed.com/exhibitors/littler-...
17,store.easternseals.co.uk,vat_kw,911470649,6273172,https://store.easternseals.co.uk/size/25-7x3-5...


In [2]:
import re
import duckdb
import pandas as pd

SEED = 42
CH_CSV = "../data/BasicCompanyDataAsOneFile-2026-09-01.csv"

df = pd.read_csv("../data/results/cc_pas2_candidati.csv", dtype=str, keep_default_na=False)
df["checksum_ok"] = df["checksum_ok"] == "True"
df["uk"] = df["uk"] == "True"
print("Candidați încărcați:", len(df), "| cu checksum OK:", df.checksum_ok.sum())

def normalizeaza_ch(x):
    x = str(x).strip().upper()
    if not x:
        return ""
    if x.isdigit():
        return x.zfill(8)
    prefix = re.match(r"[A-Z]+", x).group(0)
    return prefix + x[len(prefix):].zfill(8 - len(prefix))

valide = df[df.checksum_ok].drop_duplicates(["vat", "host"]).copy()
valide["ch_norm"] = valide.ch_in_context.apply(normalizeaza_ch)
print("Perechi unice (vat, host):", len(valide))

con = duckdb.connect()
con.register("valide", valide[["vat", "host", "ch_norm"]])
potrivire_ch = con.execute(f"""
    SELECT v.vat, v.host, v.ch_norm, ch.companyname, ch.regaddresspostcode,
           ch.companystatus, ch.accountsaccountcategory
    FROM valide v
    LEFT JOIN read_csv('{CH_CSV}', normalize_names=true, ignore_errors=true, all_varchar=true) ch
      ON ch.companynumber = v.ch_norm
    WHERE v.ch_norm <> ''
""").df()

print("Cu număr CH în context:", len(potrivire_ch))
print("…găsit în fișierul CH:", potrivire_ch.companyname.notna().sum())
pd.set_option("display.max_rows", 120)
potrivire_ch.sort_values("companyname", na_position="first")

Candidați încărcați: 165 | cu checksum OK: 121
Perechi unice (vat, host): 108
Cu număr CH în context: 74
…găsit în fișierul CH: 72


,vat,host,ch_norm,companyname,regaddresspostcode,companystatus,accountsaccountcategory
72,592950700,biology.ed.ac.uk,SC005336,NaN,NaN,NaN,NaN
73,843845601,retailrevealed.com,00454825,NaN,NaN,NaN,NaN
45,276137055,anticsonline.uk,01860713,ALAN TYNDALL LIMITED,GL10 3SX,Active,UNAUDITED ABRIDGED
0,278743063,www.altitude-lofts.co.uk,10676076,ALTITUDE LOFTS LIMITED,RG28 7ES,Active,UNAUDITED ABRIDGED
11,227952345,aridnorman.com,03686741,ARI D NORMAN LIMITED,SW19 7JY,Active,TOTAL EXEMPTION FULL
33,318223131,www.b-jewellery.co.uk,12419643,B JEWELLERY LTD,NN8 1LD,Active,MICRO ENTITY
12,408563392,flooringwarehousedirect.co.uk,13803717,BD COMMERCE LTD,HD2 1GN,Active,TOTAL EXEMPTION FULL
13,356004522,bodytransformationcentre.com,10992911,BODY TRANSFORMATION CENTRE LOUGHBOROUGH LTD,LE11 1FW,Active,MICRO ENTITY
27,215968832,www.broadwaytravel.com,00472065,BROADWAY TRAVEL SERVICE(WIMBLEDON)LIMITED,NE4 7YA,Active,FULL
52,160172148,www.bucklandtimber.co.uk,08468232,BUCKLAND TIMBER LIMITED,EX17 1DN,Active,TOTAL EXEMPTION FULL


In [3]:
pd.set_option("display.max_colwidth", 60)
print(potrivire_ch[["host", "ch_norm", "companyname", "companystatus"]].to_string())

print()
grupuri = {
    "FULL": 4, "MEDIUM": 4, "GROUP": 4, "AUDIT EXEMPTION SUBSIDIARY": 4,
    "MICRO ENTITY": 3, "TOTAL EXEMPTION FULL": 3, "UNAUDITED ABRIDGED": 3, "SMALL": 3,
    "TOTAL EXEMPTION SMALL": 3, "AUDITED ABRIDGED": 3,
    "DORMANT": 1, "FILING EXEMPTION SUBSIDIARY": 1,
}
potrivire_ch["grup"] = potrivire_ch.accountsaccountcategory.map(grupuri).fillna(2)
print(potrivire_ch.loc[potrivire_ch.companyname.notna()]
      .groupby(["grup", "accountsaccountcategory"]).size())

                             host   ch_norm                                  companyname companystatus
0        www.altitude-lofts.co.uk  10676076                       ALTITUDE LOFTS LIMITED        Active
1             thedrycleaningco.uk  15108838                    CHESHIRE DRY CLEANERS LTD        Active
2                cosmedicheck.com  13904332                             COSMEDICHECK LTD        Active
3        www.hope-education.co.uk  01135827                     FINDEL EDUCATION LIMITED        Active
4                fireangel.de.com  03641019          FIREANGEL SAFETY TECHNOLOGY LIMITED        Active
5             www.mbm-omega.co.uk  02805926                            MBM OMEGA LIMITED        Active
6       personalbesteducation.com  07889300          MOUNTBATTEN SCHOOL SERVICES LIMITED        Active
7             www.thedjshop.co.uk  03617782           MOVEMENT, SOUND & LIGHTING LIMITED        Active
8                www.pavers.co.uk  01014213                              

In [4]:
foaie = valide.merge(
    potrivire_ch[["vat", "host", "companyname", "regaddresspostcode", "accountsaccountcategory"]],
    on=["vat", "host"], how="left")
foaie = foaie.sample(frac=1, random_state=SEED).reset_index(drop=True)   # ordine aleatorie
foaie.insert(0, "ordine", range(1, len(foaie) + 1))

for col in ["hmrc_valid", "nume_hmrc", "cp_hmrc", "data_ora_bst",
            "nume_hmrc_eq_ch", "cp_hmrc_eq_ch", "tip_site", "clasificare", "note"]:
    foaie[col] = ""

coloane = ["ordine", "vat", "host", "url", "ch_norm", "companyname", "regaddresspostcode",
           "accountsaccountcategory", "hmrc_valid", "nume_hmrc", "cp_hmrc", "data_ora_bst",
           "nume_hmrc_eq_ch", "cp_hmrc_eq_ch", "tip_site", "clasificare", "note", "context"]
foaie[coloane].to_csv("../data/hmrc_checks/cc_pas2_verificare_hmrc.csv", index=False)
print(len(foaie), "rânduri salvate")

108 rânduri salvate


In [5]:
f = pd.read_csv("../data/hmrc_checks/cc_pas2_verificare_hmrc.csv", dtype=str, keep_default_na=False)
for _, r in f.iterrows():
    print(f"{r.ordine:>3}  {r.vat}  {r.companyname or '(fara CH)'}  [{r.host}]")

  1  746365902  (fara CH)  [www.warmco.co.uk]
  2  485579528  CHESHIRE DRY CLEANERS LTD  [thedrycleaningco.uk]
  3  113152467  HORNBECK LTD  [hornbeckgroup.com]
  4  442508802  HW NAIL SYSTEMS LIMITED  [hwnailsystems.co.uk]
  5  267095379  DERRADDA PROPERTIES LIMITED  [www.kingfisherhousebc.co.uk]
  6  229015134  QUITTANCE LIMITED  [www.quittance.co.uk]
  7  360367950  SECKFORD WINES LIMITED  [www.seckfordwines.co.uk]
  8  119540911  SKIRTING WORLD LTD  [mdfskirtingworld.co.uk]
  9  241344200  MARKETING SOCIETY LIMITED,(THE)  [www.marketingsociety.com]
 10  927202736  (fara CH)  [www.accreditedengineers.com]
 11  510984296  COSMEDICHECK LTD  [cosmedicheck.com]
 12  373584720  (fara CH)  [apctechnologygroup.com]
 13  197730860  (fara CH)  [www.buysheds.com]
 14  227952345  ARI D NORMAN LIMITED  [aridnorman.com]
 15  718212749  (fara CH)  [www.cochrane.org]
 16  319080665  ROBINSON WIRE CLOTH LIMITED  [www.robinsonwirecloth.co.uk]
 17  215968832  BROADWAY TRAVEL SERVICE(WIMBLEDON)LIMITED

In [6]:
f = pd.read_csv("../data/hmrc_checks/cc_pas2_verificare_hmrc.csv", dtype=str, keep_default_na=False)
t = pd.read_csv("../data/hmrc_checks/hmrc_transcriere.csv", dtype=str, keep_default_na=False).drop_duplicates("vat", keep="last")

f = f.drop(columns=["hmrc_valid", "nume_hmrc", "cp_hmrc", "data_ora_bst"]).merge(t, on="vat", how="left").fillna("")
print("Rânduri verificate la HMRC:", (f.hmrc_valid != "").sum(), "din", len(f))
print("Numere transcrise care nu sunt în listă:", sorted(set(t.vat) - set(f.vat)))
f.to_csv("../data/hmrc_checks/cc_pas2_verificare_hmrc.csv", index=False)

Rânduri verificate la HMRC: 31 din 108
Numere transcrise care nu sunt în listă: []


In [7]:
gata = f[f.nume_hmrc != ""].copy()

In [8]:
gata = f[f.hmrc_valid != ""].copy()

In [9]:
import re
import duckdb
import pandas as pd

CH_CSV = "../data/BasicCompanyDataAsOneFile-2026-09-01.csv"
f = pd.read_csv("../data/hmrc_checks/cc_pas2_verificare_hmrc.csv", dtype=str, keep_default_na=False)

def norm_nume(s):
    s = s.upper().replace("&", "AND")
    s = re.sub(r"\bLIMITED\b", "LTD", s)
    s = re.sub(r"\(THE\)|\bTHE\b", "", s)
    return re.sub(r"[^A-Z0-9]", "", s)

def norm_cp(s):
    return re.sub(r"\s", "", s.upper())

gata = f[f.hmrc_valid != ""].copy()
print("Rânduri verificate la HMRC:", len(gata), "din", len(f))

# 1. Rândurile cu număr CH în context: comparație directă
cu_ch = (gata.companyname != "") & (gata.hmrc_valid == "da")
gata.loc[cu_ch, "nume_hmrc_eq_ch"] = [
    "da" if norm_nume(a) == norm_nume(b) else "nu"
    for a, b in zip(gata.loc[cu_ch, "nume_hmrc"], gata.loc[cu_ch, "companyname"])]
gata.loc[cu_ch, "cp_hmrc_eq_ch"] = [
    "da" if norm_cp(a) == norm_cp(b) else "nu"
    for a, b in zip(gata.loc[cu_ch, "cp_hmrc"], gata.loc[cu_ch, "regaddresspostcode"])]

# 2. Rândurile valide fără număr CH: căutare după numele de la HMRC în tot fișierul CH
fara = gata[(gata.companyname == "") & (gata.hmrc_valid == "da")].copy()
fara["nume_norm"] = fara.nume_hmrc.apply(norm_nume)
con = duckdb.connect()
con.register("fara", fara[["vat", "host", "nume_hmrc", "nume_norm", "cp_hmrc"]])
gasite = con.execute(f"""
    WITH ch AS (
        SELECT companynumber, companyname, regaddresspostcode, accountsaccountcategory,
               regexp_replace(
                 regexp_replace(
                   regexp_replace(replace(upper(companyname), '&', 'AND'),
                                  '\\bLIMITED\\b', 'LTD', 'g'),
                   '\\(THE\\)|\\bTHE\\b', '', 'g'),
                 '[^A-Z0-9]', '', 'g') AS nume_norm
        FROM read_csv('{CH_CSV}', normalize_names=true, ignore_errors=true, all_varchar=true)
    )
    SELECT f.vat, f.host, f.nume_hmrc, ch.companynumber, ch.companyname AS nume_ch,
           ch.accountsaccountcategory,
           replace(upper(ch.regaddresspostcode), ' ', '') = replace(upper(f.cp_hmrc), ' ', '') AS cp_egal
    FROM fara f LEFT JOIN ch ON ch.nume_norm = f.nume_norm
""").df()
print("\nFără număr CH în context — căutare după numele HMRC:")
print(gasite.to_string())

# 3. Clasificare propusă (de verificat de mână)
def clasifica(r):
    if r.hmrc_valid == "nu":
        return "respins_inexistent"
    if r.companyname == "":
        return "de_verificat_fara_ch"
    if r.nume_hmrc_eq_ch == "nu":
        return "respins_alta_firma?"
    return "confirmat" if r.cp_hmrc_eq_ch == "da" else "incert"

gata["clasificare"] = gata.apply(clasifica, axis=1)
print("\n", gata.clasificare.value_counts())
print("\nCazuri cu nume diferit (de citit de mână):")
print(gata[gata.nume_hmrc_eq_ch == "nu"][["ordine", "vat", "host", "companyname", "nume_hmrc"]].to_string())

gata.to_csv("../data/hmrc_checks/cc_pas2_rezultat_hmrc.csv", index=False)

Rânduri verificate la HMRC: 31 din 108

Fără număr CH în context — căutare după numele HMRC:
         vat                         host                     nume_hmrc companynumber                           nume_ch accountsaccountcategory  cp_egal
0  497559419              octoclassic.com               OCTOCLASSIC LTD      12633297                   OCTOCLASSIC LTD            MICRO ENTITY     True
1  746365902             www.warmco.co.uk                NOMOCO LIMITED      01536642                    NOMOCO LIMITED      UNAUDITED ABRIDGED     True
2  197730860             www.buysheds.com           ILIKESTORES LIMITED      09220253               ILIKESTORES LIMITED            MICRO ENTITY     True
3  927202736  www.accreditedengineers.com  NEWBURY INVESTMENTS (UK) LTD      02533036  NEWBURY INVESTMENTS (UK) LIMITED                   GROUP     True
4  373584720       apctechnologygroup.com      APC TECHNOLOGY GROUP LTD      01635609          APC TECHNOLOGY GROUP LTD                   GROU

In [10]:
toate = gata[gata.hmrc_valid == "da"].copy()
toate["nume_norm"] = toate.nume_hmrc.apply(norm_nume)
con.register("toate", toate[["vat", "host", "nume_hmrc", "nume_norm", "ch_norm"]])
comp = con.execute(f"""
    WITH ch AS (
        SELECT companynumber, companyname,
               regexp_replace(regexp_replace(regexp_replace(replace(upper(companyname), '&', 'AND'),
                 '\\bLIMITED\\b', 'LTD', 'g'), '\\(THE\\)|\\bTHE\\b', '', 'g'), '[^A-Z0-9]', '', 'g') AS nume_norm
        FROM read_csv('{CH_CSV}', normalize_names=true, ignore_errors=true, all_varchar=true)
    )
    SELECT t.vat, t.host, t.nume_hmrc, t.ch_norm AS ch_context, ch.companynumber AS ch_dupa_nume
    FROM toate t LEFT JOIN ch ON ch.nume_norm = t.nume_norm
    WHERE t.ch_norm <> ''
""").df()
comp["acelasi"] = comp.ch_context == comp.ch_dupa_nume
print(comp.acelasi.value_counts(dropna=False))
print(comp[~comp.acelasi].to_string())

acelasi
True     19
False     2
Name: count, dtype: int64
          vat                     host                 nume_hmrc ch_context ch_dupa_nume  acelasi
14  433711375  www.taylormaxwell.co.uk  TAYLOR MAXWELL GROUP LTD   00476749     05726000    False
16  413203651            www.veeve.com     UNDER THE DOORMAT LTD   07666067     08968561    False


In [1]:
import re
import random
import pandas as pd

pd.set_option("display.max_colwidth", 500)

def checksum_ok(nr9):
    if len(nr9) != 9 or not nr9.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr9[:7], range(8, 1, -1))) + int(nr9[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

# 1. Checksum pe numerele confirmate din Endole
# Caut orice număr de 9 cifre din fișier (TVA = 9 cifre; numărul CH are 8 caractere)
text_endole = open("../data/hmrc_checks/urma_endole_30.csv", encoding="utf-8").read()
nr_endole = sorted(set(re.findall(r"(?<!\d)\d{9}(?!\d)", text_endole)))
print("1. Numere de 9 cifre în urma_endole_30.csv:", len(nr_endole))
for n in nr_endole:
    print("  ", n, checksum_ok(n))
print("   Toate valide:", all(checksum_ok(n) for n in nr_endole),
      "| 123456789:", checksum_ok("123456789"))

# 2 și 3. Contextul pentru Charlotte Tilbury și SC005336
cand = pd.read_csv("../data/results/cc_pas2_candidati.csv", dtype=str, keep_default_na=False)
print("\n2. Charlotte Tilbury (144073630):")
for _, r in cand[cand.vat == "144073630"].iterrows():
    print("   URL:", r.url)
    print("   CONTEXT:", r.context, "\n")

print("3. biology.ed.ac.uk (SC005336):")
for _, r in cand[cand.ch_in_context.str.upper() == "SC005336"].iterrows():
    print("   URL:", r.url)
    print("   CONTEXT:", r.context, "\n")

# 4. Sondaj pe transcriere: 4 rânduri aleatorii de reverificat de mână la HMRC
t = pd.read_csv("../data/hmrc_checks/hmrc_transcriere.csv", dtype=str, keep_default_na=False)
sondaj = t.sample(n=4, random_state=7)
print("4. Sondaj transcriere (de reverificat pe pagina HMRC):")
print(sondaj.to_string(index=False))

1. Numere de 9 cifre în urma_endole_30.csv: 0
   Toate valide: True | 123456789: False

2. Charlotte Tilbury (144073630):
   URL: https://www.charlottetilbury.com/au/secrets/how-to-apply-foundation-for-oily-skin
   CONTEXT: Ltd., trading as Charlotte Tilbury Beauty. All rights reserved. Company number 08037372, registered in the United Kingdom. Registered Office Address: 8 Surrey Street, London, United Kingdom WC2R 2ND. VAT number: GB 144 0736 30. Contact us Privacy Policy Cookies Policy Terms & Conditions Corporate Policies Manage Cookies This site is protected by reCAPTCHA and the Google Privacy Policy and Google Terms of Service apply.  

   URL: https://www.charlottetilbury.com/us-es/product/lip-lustre-hall-of-fame?from_multi_product_card=true
   CONTEXT: 6 © Charlotte Tilbury Beauty Inc., trading as Charlotte Tilbury Beauty. All rights reserved. Company number 5493834, registered in Delaware. Business Address 148 Lafayette Street, New York, NY 10013. VAT number: GB 144 0736 30. Po

In [2]:
ue = pd.read_csv("../data/hmrc_checks/urma_endole_30.csv", dtype=str, keep_default_na=False)
print(ue.columns.tolist())
print(ue.head(3).to_string())

['company_number', 'nume_ch', 'postcode_ch', 'sic', 'link_endole', 'vat_endole', 'vrn_gasit', 'rezultat_hmrc', 'nume_hmrc', 'postcode_hmrc', 'data_ora', 'categorie', 'timp_min', 'observatii']
  company_number                             nume_ch postcode_ch                                                       sic                                                                          link_endole vat_endole vrn_gasit rezultat_hmrc nume_hmrc postcode_hmrc data_ora categorie timp_min observatii
0       11906583          MOHARI SERVICES UK LIMITED     W1K 1RA                   96090 - Other service activities n.e.c.        https://open.endole.co.uk/insight/company/11906583-mohari-services-uk-limited                                                                                                  
1       14056015  BLUEPRINT PICTURES (STRANGERS) LTD    WC1B 5EH              59111 - Motion picture production activities  https://open.endole.co.uk/insight/company/14056015-blueprint-pictures-st

In [3]:
for col in ["vat_endole", "vrn_gasit", "rezultat_hmrc"]:
    print(col, "| completate:", (ue[col].str.strip() != "").sum())

numere = (ue["vat_endole"] + " " + ue["vrn_gasit"]).str.replace(r"\D", " ", regex=True)
nr_endole = sorted({n for s in numere for n in re.findall(r"\d{9}", s.replace(" ", "")) if len(n) == 9})
print("\nNumere găsite:", len(nr_endole))
for n in nr_endole:
    print("  ", n, checksum_ok(n))
print("Toate valide:", len(nr_endole) > 0 and all(checksum_ok(n) for n in nr_endole),
      "| 123456789:", checksum_ok("123456789"))

vat_endole | completate: 0
vrn_gasit | completate: 0
rezultat_hmrc | completate: 0

Numere găsite: 0
Toate valide: False | 123456789: False
